# FedSwarm — DAY 1: the gate, then A2 (Kaggle GPU)

Everything comes from GitHub. **There is exactly one thing you must click**, because the MRI
images are not in the repository and should not be (7,200 JPEGs, and the licence is the
dataset's, not ours):

> **Right sidebar → + Add Input → Datasets → search `masoudnickparvar/brain-tumor-mri-dataset`
> → Add.**
> Then **Notebook options → Accelerator → GPU T4 x2**.

Nothing is downloaded to your computer. Cell 1 clones the repo; the dataset mounts read-only
under `/kaggle/input/`.

## What this notebook does, and why in this order

| step | what | cells | time |
|---|---|---|---|
| 1 | **`gate_fitness`** — which fitness fix closes the degenerate optimum | 8 | **~15 min** |
| 2 | **Read the verdict and apply it** — patches the one lever every later sweep inherits | — | 1 min |
| 3 | **`a2_reduced`** — does cross-round pheromone memory help at all | 30 | **~6 GPU-h** |

**Why A2 today rather than the main sweep.** Resolving the paper's citations on 2026-09-25
found that FedAWA (CVPR 2025), Adp-FL-PSO and FedPSO all already optimise aggregation weights —
and all three are **stateless between rounds**. Cross-round pheromone persistence is the only
structural novelty this project has left, and A2 is the only experiment that tests it. 6
GPU-hours decide whether there is a contribution at all, so they come before 30 GPU-hours of
headline table.

## The two outcomes that should stop you

- **Step 2 says no arm closed the corner.** Do not run step 3. Every search method would
  inherit the same degenerate optimum. Raise `aco-gamma-entropy` past 0.45 and re-run step 1.
- **Step 3 shows `none` ≈ `decayed` ≈ `full`.** Persistence buys nothing, so the last
  structural novelty is gone. Stop and re-plan before spending the remaining ~64 GPU-hours —
  the right move then is a shorter methods paper, not more sweeps.

Read `HANDOVER.md` in the repo for full context.

## 1. Setup — clone from GitHub

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/researchpaper784-alt/ResearchPaper.git"
REPO_DIR = "/kaggle/working/ResearchPaper"

# BRANCH is not optional. A bare `git clone` takes the repository's DEFAULT branch, which is
# `main`, and every config this notebook runs -- gate_fitness.yaml, ablation_a2_reduced.yaml,
# scripts/apply_gate_fix.py -- exists only on the feature branch. On a `main` checkout the
# cells below fail with "no such config", which reads like a broken notebook.
BRANCH = "claude/happy-hamilton-c5jjil"

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

on = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                    capture_output=True, text=True).stdout.strip()
print("branch:", on)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout)
if on != BRANCH:
    raise SystemExit(f"Checked out {on!r}, not {BRANCH!r}.")

In [ ]:
%cd /kaggle/working/ResearchPaper

# flwr[simulation] pulls in ray; installed first and on its own. fedswarm is --no-deps because
# its own pins (torch==2.2.2, numpy<2) target the dev machine and have no Kaggle CUDA build --
# Kaggle's base image already has a newer working torch/numpy.
!pip install -q "flwr[simulation]>=1.36.0,<1.37.0"
!pip install -q --no-deps -e .
!pip install -q omegaconf rich

In [ ]:
import glob
import os
import sys
from pathlib import Path

inputs = sorted(glob.glob("/kaggle/input/*"))
print("inputs mounted:", [Path(p).name for p in inputs] or "NONE")
if not inputs:
    raise SystemExit(
        "No dataset under /kaggle/input/. Fix: right sidebar -> + Add Input -> Datasets -> "
        "search masoudnickparvar/brain-tumor-mri-dataset -> Add."
    )

# Pick the input that actually holds the images rather than trusting glob order: from session 2
# onward you will have added this notebook's own previous output back in, and inputs[0] would
# then point the data root at a results folder.
DATA_ROOT = next((p for p in inputs if any(Path(p).rglob("Training"))), None)
if DATA_ROOT is None:
    raise SystemExit(
        "None of the mounted inputs contains a Training/ directory, so none is the MRI "
        f"dataset. Mounted: {[Path(p).name for p in inputs]}."
    )
print("dataset root:", DATA_ROOT)

# `flwr run` executes an INSTALLED COPY of the app, whose __file__ is not this clone, so the
# app resolves data and cache paths against FEDSWARM_REPO_ROOT rather than its own location.
os.environ["FEDSWARM_DATA_ROOT"] = DATA_ROOT
os.environ["FEDSWARM_REPO_ROOT"] = "/kaggle/working/ResearchPaper"
os.environ["FLWR_DISABLE_RUNTIME_DEPENDENCY_INSTALLATION"] = "1"

sys.path.insert(0, "src")
import torch  # noqa: E402
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU visible. Step 3 is ~6 GPU-hours and will not finish on CPU. Fix: Notebook "
        "options -> Accelerator -> GPU T4 x2, then Run -> Restart & clear cell outputs."
    )
print("CPU cores:", os.cpu_count())

import flwr  # noqa: E402
from fedswarm.data.download import find_split_parent  # noqa: E402
print("flwr:", flwr.__version__, "| torch:", torch.__version__)
print("split parent:", find_split_parent(Path(DATA_ROOT)))

### Restore results from a previous session

Skip on your first run. From session 2 onward, add this notebook's own previous output as an
input (**+ Add Input → Your Work → Notebook Output**) and this restores it, so every sweep
resumes per-cell instead of starting over.

In [ ]:
import shutil

RESULTS = Path("/kaggle/working/ResearchPaper/results")
RESULTS.mkdir(parents=True, exist_ok=True)

restored = 0
for prior in glob.glob("/kaggle/input/**/results", recursive=True):
    for src in Path(prior).rglob("*.json*"):
        dst = RESULTS / src.relative_to(prior)
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
            restored += 1
print(f"restored {restored} file(s) from previous sessions")
print("result files now present:", len(list(RESULTS.rglob('*.json'))))

## 2. Build the image cache (~3 min, once per session)

In [ ]:
import pandas as pd

from fedswarm.data.cache import build_and_save_cache
from fedswarm.data.download import find_split_parent, resolve_root

manifest = pd.read_csv("data/processed/manifest.csv")
print("manifest rows:", len(manifest), "| pseudo-patients:", manifest.pseudo_patient_id.nunique())
cache_path = build_and_save_cache(manifest, find_split_parent(resolve_root(None)), 112)
print("cache:", cache_path)

## 3. Federation and GPU — **not optional**

Ray hides the GPU from any actor requested with `num_gpus=0`, so with no fraction set every
client trains on **CPU** while the server keeps the card — and every logged metric looks
completely normal. The only symptom is wall-clock. `1/num_clients` lets all ten share one T4.

In [ ]:
GPUS_PER_CLIENT = 0.1   # 1/10 clients
print("GPUs per ClientApp:", GPUS_PER_CLIENT)

---
# STEP 1 — the gate: which fitness fix closes the degenerate optimum

8 cells, 120 rounds, **~15 minutes.**

The problem this settles, from `docs/OPEN_QUESTIONS.md`: on the first real GPU run the best
**single-client vertex** — put all the weight on one client and discard the rest — outscored the
FedAvg reference point in **15 of 15 rounds**, mean margin **+0.6252**. The colony did not go
there only because the search was too short to find it, so every health signal read normal while
the objective pointed somewhere useless. **The failure arrives as the search gets better.**

Two candidate fixes exist and neither has ever been confirmed on real data:

| arm | what it changes | evidence so far |
|---|---|---|
| `default` | nothing — reproduces the failure | corner won **15/15**, margin **+0.6252** |
| `gamma_entropy_0.45` | raises the concentration penalty to its closed-form crossing | a **lower** bound; 0.6 closed the corner and broke everything downstream |
| `dispersion_aggregate` | a different dispersion shape | on the local fixture: corner won **0/15**, margin **−0.5283**, every round deposited |
| `fedavg` | — | gives health check 4 the baseline it lacked |

**Read `corner_margin`, not macro-F1.** 15 rounds is far too short for macro-F1, and reading it
is how the previous round of this argument went wrong.

In [ ]:
# 8 cells x 15 rounds. Resumes per cell, so a re-run costs nothing already done.
!python scripts/run_sweep_granular.py --config configs/experiment/gate_fitness.yaml --gpus-per-client {GPUS_PER_CLIENT}

In [ ]:
# The four health questions, on the gate's own directory.
!python scripts/check_fedaco_health.py --results-dir results/fl/gate

---
# STEP 2 — the verdict, and applying it

`apply_gate_fix.py` prints each arm's per-round corner margin and picks a winner. The rule is
**negative in every round, not on average**: the penalty weight and the dispersion shape are set
once per run, so a value that clears the mean round leaves the worst rounds degenerate.

The next cell only *reports*. Nothing is changed yet.

In [ ]:
!python scripts/apply_gate_fix.py --from-results results/fl/gate

### Apply it

This patches **`pyproject.toml`'s `[tool.flwr.app.config]`**, which is the one lever every later
sweep inherits. That target matters: `ablation_a1_reduced` and `ablation_a2_reduced` set
`strategy-name: fedaco` in `base_overrides` and **never read `configs/strategy/fedaco.yaml`** —
patching that file would fix the sweeps using `file:` and silently leave the two that decide the
paper running the broken default.

**If the cell above said no arm closed the corner, stop here.** The cell below will refuse, and
it is right to: running A2 or A1 on an open corner buys a number that cannot be interpreted.

In [ ]:
!python scripts/apply_gate_fix.py --from-results results/fl/gate --apply-verdict

# Confirm the change is what the runner will actually read, rather than trusting the message.
import tomllib
with open("pyproject.toml", "rb") as fh:
    live = tomllib.load(fh)["tool"]["flwr"]["app"]["config"]
print("\nlive config now:")
for key in ("aco-gamma-entropy", "aco-dispersion-reference", "aco-q0", "aco-rho-round"):
    print(f"  {key:28} = {live[key]!r}")

---
# STEP 3 — A2: does cross-round pheromone memory do anything?

30 cells, 3,000 rounds, **~6 GPU-h.** Three arms — `none`, `decayed`, `full` — across
Dirichlet 0.3 and 0.1, 5 seeds.

**This is the experiment carrying the paper's contribution.** Not because it was designed to,
but because the citation search found FedAWA (CVPR 2025), Adp-FL-PSO and FedPSO already
optimising aggregation weights — all of them stateless between rounds. Persistence is what is
left that nobody else has.

A Kaggle session caps around 9 hours, so this fits in one with room. If it does get cut off,
re-run the notebook: cell 6 restores the finished cells and the sweep resumes.

In [ ]:
!python scripts/run_sweep_granular.py --config configs/experiment/ablation_a2_reduced.yaml --gpus-per-client {GPUS_PER_CLIENT}

In [ ]:
# The table. `decayed` is the project default, so it is the middle of the three.
!python scripts/make_tables.py --results-dir results/fl/ablation --out paper/tables

# And the direct question, read off the results rather than the table's formatting.
import json
from collections import defaultdict

by_arm = defaultdict(list)
for path in sorted(Path("results/fl/ablation").rglob("*.json")):
    r = json.loads(path.read_text())
    cfg, final = r.get("config") or {}, r.get("final") or {}
    mode, f1 = cfg.get("aco-persistence"), final.get("test_macro_f1")
    if mode and f1 is not None:
        by_arm[(mode, cfg.get("alpha"))].append(f1)

print(f"\n{'persistence':14} {'alpha':>6} {'n':>3} {'mean macro-F1':>14} {'std':>8}")
for (mode, alpha) in sorted(by_arm, key=lambda k: (str(k[1]), k[0])):
    vals = by_arm[(mode, alpha)]
    mean = sum(vals) / len(vals)
    std = (sum((v - mean) ** 2 for v in vals) / max(1, len(vals) - 1)) ** 0.5
    print(f"{mode:14} {str(alpha):>6} {len(vals):>3} {mean:>14.4f} {std:>8.4f}")

print("\nIf none / decayed / full sit inside each other's std, persistence buys nothing and")
print("the last structural novelty is gone. Report that before spending the other ~64 GPU-h.")

---
# 4. Before the session ends — SAVE, or you lose everything

Kaggle discards `/kaggle/working` unless the notebook is **committed**. Use
**Save Version → Save & Run All (Commit)**, not the quick save.

Then, for the next session, add this notebook's output back as an input so cell 6 restores it.

In [ ]:
results = sorted(Path("results").rglob("*.json"))
print(f"{len(results)} result file(s) to save")
for directory in sorted({p.parent for p in results}):
    print(f"  {directory}: {len(list(directory.glob('*.json')))}")

print("\nNext: Save Version -> Save & Run All (Commit).")
print("Then Day 2 is `a1-reduced` (80 cells, ~17 GPU-h) -- the go/no-go on the framing.")